# Part 2: Belief States

After observing a sequence $x_{1:t}$, the observer's optimal summary of the past is the **belief state**: the posterior distribution over hidden states,
$$
\eta^{(x_{1:t})}_j \;=\; P\big(s_t = j \mid x_{1:t}\big), \qquad \eta^{(x_{1:t})} \in \Delta^{n_\text{states}-1}.
$$
This is a vector of shape `(n_states,)` lying on the probability simplex. The unnormalized propagated vector from Part 1, $v = \text{initial\_vect}\cdot T[x_1]\cdots T[x_t]$, already has entries $v_j = P(x_{1:t}, s_t = j)$; normalizing by $\sum_j v_j = P(x_{1:t})$ turns the joint into the posterior. So the belief state is just a *renormalized* propagated vector — but that renormalization is exactly what makes it the right state for prediction.

In this part the non-belief machinery (`_propagate`, `sequence_probability`, `conditional_next_token_probabilities`, `all_next_token_probabilities`) is **provided pre-implemented** in the HMM skeleton; you may call it freely. You will implement, in order:

1. `belief_state` — the posterior distribution $\eta^{(x_{1:t})}$ over hidden states given the observed sequence $x_{1:t}$.
2. `belief_update` — the recursive one-symbol Bayesian update $\eta^{(x_{1:t})} \mapsto \eta^{(x_{1:t+1})}$, avoiding re-propagation from scratch.
3. `ntp_from_belief_state` — the next-token distribution implied by a belief state.
4. `all_belief_states` — every reachable belief state up to a given depth, the data needed to draw the mixed-state simplex.

The workflow is identical to Part 1: write the method, monkeypatch it onto your `HMM` class, and check it with `tests.test_<method>(HMM)`.

## Setup

In [ ]:
# Best-effort auto-reload of edited modules (silently skipped where IPython's
# autoreload extension is unavailable, e.g. some Colab Python 3.12 images).
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

import sys
import pathlib

# Make the exercise modules (processes / solutions / tests / plotting) importable
# regardless of the kernel's working directory (the exercises folder or the repo root).
_here = pathlib.Path.cwd()
for _cand in [_here, _here / "exercises"]:
    if (_cand / "tests.py").exists():
        sys.path.insert(0, str(_cand))
        break

import numpy as np
from collections.abc import Sequence

import processes
import solutions
import tests

# Shape aliases (documentation only -- pyright sees plain np.ndarray).
type TransitionTensor = np.ndarray  # (n_obs, n_states, n_states)
type StateVector = np.ndarray       # (n_states,) -- belief / propagated state vector
type TokenDist = np.ndarray         # (n_obs,)    -- distribution over next tokens

---

## Exercises

The cells above are **setup**. First run the cell below to define the `HMM` class, then work through the exercises in order: for each, complete the `# YOUR CODE HERE` cell (it attaches the method to `HMM` via `HMM.<name> = <name>`), then run its `tests.test_<name>(HMM)` cell — it prints **All tests passed!** when your implementation is correct.

The `HMM` skeleton — each exercise adds a method to this class via `HMM.<name> = <name>`.

In [ ]:
# The non-belief-state methods from Part 1 are GIVEN here so this notebook is
# standalone. You implement only the belief-state methods below.

class HMM:
    """HMM defined by an observation-indexed transition tensor and an initial vector.

    Shapes
    ------
    transition_tensor        : (n_obs, n_states, n_states)
    belief / state vectors   : (n_states,)
    next-token distributions : (n_obs,)
    """

    def __init__(self, transition_tensor: TransitionTensor, initial_vect: StateVector) -> None:
        self.transition_tensor = np.asarray(transition_tensor)
        self.initial_vect = np.atleast_1d(np.asarray(initial_vect)).ravel()

    def _propagate(self, sequence: Sequence[int], vect: StateVector | None = None) -> StateVector:
        """Run a sequence through the tensor, returning the unnormalized state vector."""
        vect = self.initial_vect if vect is None else vect
        for obs in sequence:
            vect = np.einsum("i,ij->j", vect, self.transition_tensor[obs])
        return vect

    def sequence_probability(self, sequence: Sequence[int]) -> float:
        return np.sum(self._propagate(sequence))

    def conditional_next_token_probabilities(self, sequence: Sequence[int]) -> TokenDist:
        seq = list(sequence)
        p_seq = self.sequence_probability(seq)
        if np.allclose(p_seq, 0.0):
            raise ValueError(f"sequence {tuple(sequence)} has zero probability; "
                             "next-token distribution is undefined.")
        n_obs = self.transition_tensor.shape[0]
        joint = np.array([self.sequence_probability(seq + [k]) for k in range(n_obs)])
        return joint / p_seq

    def all_next_token_probabilities(self, max_depth: int) -> dict[tuple[int, ...], TokenDist]:
        """Next-token distribution for every reachable sequence of length 0..max_depth.

        Built breadth-first; a continuation is reachable iff its probability in the
        current distribution is nonzero (zero-probability branches are pruned).
        Uses conditional_next_token_probabilities directly -- no belief-state machinery.
        Includes () -> the prior next-token distribution.
        """
        prior = self.conditional_next_token_probabilities(())
        result = {(): prior}
        frontier = [((), prior)]
        for _ in range(max_depth):
            next_frontier = []
            for seq, ntp in frontier:
                for obs in range(len(ntp)):
                    if np.allclose(ntp[obs], 0.0):
                        continue  # zero-probability continuation -> prune
                    new_seq = seq + (obs,)
                    child = self.conditional_next_token_probabilities(new_seq)
                    result[new_seq] = child
                    next_frontier.append((new_seq, child))
            frontier = next_frontier
        return result

### Exercise - implement `belief_state`

> **Difficulty:** 🔴🔴⚪⚪⚪  
> **Importance:** 🔵🔵🔵🔵⚪  
> 
> You should spend up to ~10 minutes on this exercise.

The belief state is the observer's posterior over the current hidden state given everything seen so far: $\eta^{(x_{1:L})}_j = P(\text{state} = j \mid x_{1:L})$, a vector of shape `(n_states,)` that sums to 1. Where `_propagate` returns the *unnormalized* forward vector $\alpha^{(x_{1:L})}$, with $\alpha^{(x_{1:L})}_j = P(x_{1:L},\ \text{state} = j)$, the belief state is just $\alpha$ rescaled to a probability distribution.

By the definition of conditional probability,
$$
\eta^{(x_{1:L})}_j = \frac{P(x_{1:L},\ \text{state}=j)}{P(x_{1:L})} = \frac{\alpha^{(x_{1:L})}_j}{\sum_k \alpha^{(x_{1:L})}_k}.
$$
The normalizer $\sum_k \alpha^{(x_{1:L})}_k$ is exactly `sequence_probability(sequence)`. If it is zero the sequence is impossible under the model, so no posterior exists — raise `ValueError` in that case rather than dividing by zero.

This normalization is what makes belief states a finite-memory sufficient statistic: the scalar magnitude of $\alpha^{(x_{1:L})}$ (the running joint probability) is discarded, and the resulting unit vector carries all the information from $x_{1:L}$ needed to predict the future.

<details><summary>Hint</summary>

Call `self._propagate(sequence)` to get $\alpha^{(x_{1:L})}$, take its sum, and return $\alpha^{(x_{1:L})}$ divided by that sum. Guard the sum against zero (e.g. `np.allclose(norm, 0.0)`) before dividing and raise `ValueError` for impossible sequences.

</details>

In [ ]:
def belief_state(self, sequence: Sequence[int]) -> StateVector:
    vect = self._propagate(sequence)
    norm = np.sum(vect)
    if np.allclose(norm, 0.0):
        raise ValueError(f"sequence {tuple(sequence)} has zero probability; belief state is undefined.")
    return vect / norm

HMM.belief_state = belief_state

In [ ]:
tests.test_belief_state(HMM)

### Exercise - implement `belief_update`

> **Difficulty:** 🔴🔴⚪⚪⚪  
> **Importance:** 🔵🔵🔵⚪⚪  
> 
> You should spend up to ~10-15 minutes on this exercise.

Instead of calculating the belief state for every sequence, it is convenient to maintain a belief and update it after each new observation. Given the current belief `belief_state` (shape `(n_states,)`) and a single observation `obs`, return the updated belief over the *next* hidden state.

The belief updating expression is given by:

$$
\eta^{(x_{1:t})} \mapsto \eta^{(x_{1:t+1})} = \frac{\eta^{(x_{1:t})}T^{(x_{t+1})}}{\eta^{(x_{1:t})}T^{(x_{t+1})}\mathbf{1}}
$$

If the normalizer is $0$, the observation is impossible under the current belief and there is no valid posterior — raise a `ValueError` in that case rather than dividing by zero.

<details><summary>Hint</summary>

Compute the unnormalized update with a single contraction over the source state:

```python
np.einsum("i,ij->j", belief_state, self.transition_tensor[obs])
```

Then check whether the result sums to (near) zero — raise `ValueError` if so — otherwise divide by its sum so the returned vector sums to 1.

</details>

In [ ]:
def belief_update(self, belief_state: StateVector, obs: int) -> StateVector:
    update = np.einsum("i,ij->j", belief_state, self.transition_tensor[obs])
    norm = np.sum(update)
    if np.allclose(norm, 0.0):
        raise ValueError(f"observation {obs} has zero probability from this belief state; cannot update.")
    return update / norm

HMM.belief_update = belief_update

In [ ]:
tests.test_belief_update(HMM)

### Exercise - implement `ntp_from_belief_state`

> **Difficulty:** 🔴🔴🔴⚪⚪  
> **Importance:** 🔵🔵🔵⚪⚪  
> 
> You should spend up to ~10-15 minutes on this exercise.

A belief state $\eta^{(x_{1:t})} \in \mathbb{R}^{n_\text{states}}$ summarizes everything the process knows about the latent state given the tokens seen so far: $\eta^{(x_{1:t})}_i = P(\text{state} = i \mid x_{1:t})$, with $\sum_i \eta^{(x_{1:t})}_i = 1$. This method maps that belief directly to a distribution over the next emitted symbol, with no reference to any particular sequence.

The observation-indexed transition tensor $T$ has shape $(n_\text{obs}, n_\text{states}, n_\text{states})$, where $T[k, i, j] = P(\text{emit } k,\ j \mid i)$. Marginalizing the next-state index $j$ gives the conditional probability of emitting $k$ given being in state $i$. Weighting by the belief and summing over the current state $i$ yields

$$
P(\text{next} = k \mid x_{1:t}) = \eta^{(x_{1:t})} T[k] \mathbf{1} =  \sum_{i, j} \eta^{(x_{1:t})}_i \, T[k, i, j].
$$

Because $\eta^{(x_{1:t})}$ sums to $1$ and $T$ is properly normalized (summing $T$ over $k$ and $j$ gives a row-stochastic state-transition matrix), the resulting length-$n_\text{obs}$ vector is already a valid probability distribution — no renormalization needed.

<details><summary>Hint</summary>

Contract the belief against the current-state axis of $T$ and sum out the next-state axis in one shot:

```python
np.einsum("i,kij->k", belief_state, self.transition_tensor)
```

The output is indexed by $k$ (the symbol), giving the token distribution directly.

</details>

In [ ]:
def ntp_from_belief_state(self, belief_state: StateVector) -> TokenDist:
    return np.einsum("i,kij->k", belief_state, self.transition_tensor)

HMM.ntp_from_belief_state = ntp_from_belief_state

In [ ]:
tests.test_ntp_from_belief_state(HMM)

### Exercise - implement `all_belief_states`

> **Difficulty:** 🔴🔴🔴⚪⚪  
> **Importance:** 🔵🔵⚪⚪⚪  
> 
> You should spend up to ~15 minutes on this exercise.

This method enumerates the belief state for every reachable observation sequence of length $0$ through `max_depth`, returning a dict keyed by the sequence tuple. The empty key `()` maps to the prior belief `belief_state(())`, and each key $(x_1, \dots, x_k)$ maps to the posterior over hidden states given that the process has emitted exactly that sequence.

This is the **mixed-state presentation** of the process: rather than tracking the hidden Markov state, we track the observer's belief (a distribution over states) and how it evolves as observations arrive. Each emitted symbol drives a Bayesian update $b \mapsto \text{belief\_update}(b, x)$, so the set of reachable beliefs forms the points of the mixed-state space. For processes like Mess3 this set is uncountable in the limit, and its finite-depth approximation traces out a fractal in the probability simplex.

Build the dict via breadth-first search. Start from `belief_state(())` keyed by `()`. To expand a node $(x_1,\dots,x_k)$ at depth $<$ `max_depth`, try each observation $x$: the continuation is reachable iff $P(x \mid b) > 0$, which `belief_update` signals by *not* raising. Zero-probability continuations are pruned, so the keys of the returned dict are exactly the sequences the process can actually produce.

<details><summary>Hint</summary>

No einsum here. Do a BFS over sequence tuples: maintain a frontier of `(seq, belief)` pairs, seed it with `((), belief_state(()))`, and for each frontier node at depth less than `max_depth` loop over all observations, calling `belief_update`. Wrap that call in `try/except ValueError` — a raise means the continuation has zero probability, so skip it; otherwise record `seq + (x,)` with its updated belief and push it onto the next frontier.

</details>

In [ ]:
def all_belief_states(self, max_depth: int) -> dict[tuple[int, ...], StateVector]:
    """Belief state for every reachable sequence of length 0..max_depth.

    Returns a dict mapping each observation sequence (as a tuple) to its
    belief state, starting from the empty sequence () -> prior belief.
    Built breadth-first via belief_update; zero-probability continuations
    are pruned (they have no defined belief state).
    """
    n_obs = self.transition_tensor.shape[0]
    prior = self.belief_state(())
    result = {(): prior}
    frontier = [((), prior)]
    for _ in range(max_depth):
        next_frontier = []
        for seq, belief in frontier:
            for obs in range(n_obs):
                try:
                    updated = self.belief_update(belief, obs)
                except ValueError:
                    continue  # zero-probability continuation -> prune
                new_seq = seq + (obs,)
                result[new_seq] = updated
                next_frontier.append((new_seq, updated))
        frontier = next_frontier
    return result

HMM.all_belief_states = all_belief_states

In [ ]:
tests.test_all_belief_states(HMM)

## Demo

With the belief-state methods implemented, we can collect the reachable belief states and view their geometry interactively (via the local `plotting.py`). Belief states live on the **hidden-state** simplex, so the **arch** process (4 states) gives a 3D tetrahedron; hover a point to see the inducing sequence.

In [ ]:
hmm = HMM(processes.mess3(0.15, 0.2), processes.uniform_initial(3))

beliefs = hmm.all_belief_states(5)
print("# distinct reachable sequences <= depth 5:", len(beliefs))
print("belief after observing (0, 1, 2):", np.round(hmm.belief_state((0, 1, 2)), 4))
print("next-token dist from that belief :", np.round(hmm.ntp_from_belief_state(hmm.belief_state((0, 1, 2))), 4))

In [ ]:
# The arch process: 4 hidden states, 3 symbols. Belief states lie on the 4-state
# simplex, so they are plotted in 3D (a tetrahedron).
import plotting

arch_hmm = HMM(processes.arch(0.6), processes.uniform_initial(4))
depth = 7
arch_beliefs = arch_hmm.all_belief_states(depth)
plotting.plot_belief_states(
    np.array([np.real(v) for v in arch_beliefs.values()]),
    sequences=list(arch_beliefs.keys()),
    title=f"Arch belief-state geometry (depth {depth})",
)